In [11]:
!pip install nlpaug swifter tqdm

     ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
     -------- ------------------------------- 0.3/1.2 MB ? eta -:--:--
     ----------------- ---------------------- 0.5/1.2 MB 1.5 MB/s eta 0:00:01
     -------------------------- ------------- 0.8/1.2 MB 1.5 MB/s eta 0:00:01
     ---------------------------------------- 1.2/1.2 MB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   -------------- ------------------------- 0.5/1.4 MB 1.4 MB/s eta 0:00:01
   --------------------- ------------------ 0.8/1.4 MB 1.4 MB/s eta 0:00:01
   ----------------------------- ---------- 1.0/1.4 MB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 1.4 MB/s eta 0:00:00
  Created wheel for swifter: filename=swifter-1.4.0-py3-none-any.whl size=16564 

In [1]:
import pandas as pd
import swifter
from tqdm.notebook import tqdm
import nlpaug.augmenter.char as nac

In [5]:
# Load dataset
df = pd.read_csv("../dataset/sms/train.csv")
df.head()

,email,target
0,What to think no one saying clearly. Ok leave ...,ham
1,"FREE RING TONE just text \POLYS\"" to 87131. Th...",spam
2,"Trust me. Even if isn't there, its there.",ham
3,Hi dear we saw dear. We both are happy. Where ...,ham
4,"URGENT, IMPORTANT INFORMATION FOR O2 USER. TOD...",spam


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3565 entries, 0 to 3564
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   email   3565 non-null   object
 1   target  3565 non-null   object
dtypes: object(2)
memory usage: 55.8+ KB


In [7]:
# Create a character swap augmenter with a moderate swap probability
char_aug = nac.RandomCharAug(action="swap", aug_char_p=0.20)

In [8]:
def augment_if_spam(row):
    """
    Applies a character-level swap attack using nlpaug to spam emails only.

    Parameters:
        row (pd.Series): A row from the DataFrame with 'email' and 'target'.

    Returns:
        pd.Series: (augmented_text, was_augmented)
    """
    email = row.get('email', '')

    # Only augment if it's labeled spam and is a non-empty string
    if row.get('target') == 'spam' and isinstance(email, str) and email.strip():
        try:
            augmented = char_aug.augment(email)
            return pd.Series([augmented, True])
        except Exception as e:
            print(f"Augmentation failed: {e}")
            return pd.Series([email, False])
    else:
        return pd.Series([email, False])


In [9]:
tqdm.pandas()  # Enables progress bar
df[['email_charswapped', 'was_augmented']] = df.swifter.apply(augment_if_spam, axis=1)

Pandas Apply:   0%|          | 0/3565 [00:00<?, ?it/s]

In [10]:
df.head(10)

,email,target,email_charswapped,was_augmented
0,What to think no one saying clearly. Ok leave ...,ham,What to think no one saying clearly. Ok leave ...,False
1,"FREE RING TONE just text \POLYS\"" to 87131. Th...",spam,"[FERE RING TOEN juts text \ POLYS \ "" to 87113...",True
2,"Trust me. Even if isn't there, its there.",ham,"Trust me. Even if isn't there, its there.",False
3,Hi dear we saw dear. We both are happy. Where ...,ham,Hi dear we saw dear. We both are happy. Where ...,False
4,"URGENT, IMPORTANT INFORMATION FOR O2 USER. TOD...",spam,"[UGRNET, IMPROATNT NFIORMATOIN FOR O2 SUER. TD...",True
5,Yeah there's barely enough room for the two of...,ham,Yeah there's barely enough room for the two of...,False
6,Jus telling u dat i'll b leaving 4 shanghai on...,ham,Jus telling u dat i'll b leaving 4 shanghai on...,False
7,Am slow in using biola's fne,ham,Am slow in using biola's fne,False
8,Your gonna be the death if me. I'm gonna leave...,ham,Your gonna be the death if me. I'm gonna leave...,False
9,"Good morning, my boytoy! How's those yummy lip...",ham,"Good morning, my boytoy! How's those yummy lip...",False


In [112]:
df.to_csv("dataset/sms/charswap/train_with_charswap.csv", index=False)